# 30 · Fine-tuning судьи Qwen2.5-7B по фолдам

Переписанная версия `qwen7b_full_finetune_datasphere.ipynb`. Что изменилось и почему:

| Что | Было | Стало |
|---|---|---|
| Ветка клона | `qwen7b-notebook` | `integration` |
| Сплит | `split_samples(samples, seed=SEED)` | `data/splits/folds_alfa.json` |
| `SAVE_STRATEGY` | `"no"` | `"epoch"` |
| Баланс классов | несбалансированно | взвешенный лосс + oversampling негативов |
| Выход | генеративный вердикт | вердикт + logprob → `scores.jsonl` |
| Контроль | нет | `degenerate_rate` после каждой эпохи |
| Где логика | в ячейках | в `scripts/train_ft_judge.py` |

Старый сплит — протокол A: 24.9% тестовых строк делили вопрос с train. Любое число
FT-судьи, полученное им, завышено и несравнимо с числами на group-сплитах.

**Конфигурация: `g2.1` (1× A100 80 GB).** Один фолд — ~1.5 ч, все пять — ~7.5 ч,
поэтому полный набор фолдов идёт через `jobs/ft_judge_fold{0..4}.yaml`, а здесь —
смоук и одиночный фолд.

> **Требуется от других:** `scripts/train_ft_judge.py` в репозитории ещё нет.
> Обучение в ячейках запрещено правилом карточки D3 §1 — ноутбук вызывает CLI,
> и отсутствие CLI это баг CLI. Требуемый контракт описан в `docs/datasphere.md`.

## 0. Конфигурация

In [ ]:
# ======================= КОНФИГУРАЦИЯ — правится только здесь =======================
BASE     = "/home/jupyter/filestore/neurodrive"   # File Storage: переживает рестарт VM
REPO     = f"{BASE}/rag-reliability"
REPO_URL = "https://github.com/MurkaSelebry/rag-reliability.git"
BRANCH   = "integration"                          # НЕ qwen7b-notebook: та ветка устарела
CACHE    = f"{BASE}/cache/m3_judge"               # кэш судьи -> прогон резюмируется
LOGS     = f"{BASE}/logs"
DATA     = "data/alfa.jsonl"                      # канонический корпус, 2233 кейса
FOLDS    = "data/splits/folds_alfa.json"          # сплит только отсюда, split_samples не вызываем
MODEL    = "Qwen/Qwen2.5-7B-Instruct"
API_BASE = "http://localhost:8000/v1"
# ===================================================================================

# ---- гиперпараметры обучения; все уходят в CLI флагами, ни один не читается кодом ячейки ----
MODE            = "direct"      # direct (Метод 1) или marker (Метод 2)
FOLD            = 0             # номер фолда из folds.json
REPEAT          = 0             # номер повтора 5x5 CV
TUNING          = "lora"        # lora | full
LORA_R          = 256           # все линейные слои, включая MLP
LORA_ALPHA      = 512
LEARNING_RATE   = 2e-4          # ~10x выше, чем для full FT
EPOCHS          = 3
MAX_SEQ_LEN     = 2048
BATCH_SIZE      = 1
GRAD_ACCUM      = 8
POS_WEIGHT_MODE = "balanced"    # взвешенный лосс: прошлый прогон 1.5B схлопнулся в (1,1)
SEED            = 42
SAVE_STRATEGY   = "epoch"       # НЕ "no": при "no" обрыв сессии стоит всего прогона
SAVE_TOTAL_LIMIT = 2

PUSH_TO_HUB  = False
HUB_MODEL_ID = ""               # например "your-login/qwen2.5-7b-rag-judge-fold0"
HF_TOKEN     = ""               # write-токен, https://huggingface.co/settings/tokens

import os, subprocess

branch = subprocess.check_output(
    ["git", "-C", REPO, "rev-parse", "--abbrev-ref", "HEAD"]
).decode().strip()
assert branch == BRANCH, (
    f"репозиторий на ветке {branch}, ожидалась {BRANCH}. Прогон с другой ветки несравним "
    "с остальными: перезапусти notebooks/00_setup.ipynb"
)
print("branch:", branch)

os.environ["HF_HOME"] = f"{BASE}/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
OUTPUT_DIR = f"{BASE}/ft_judge/{MODE}_fold{FOLD}"
ARTIFACT = f"predictions/alfa/ft_judge/{MODE}_fold{FOLD}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("output:", OUTPUT_DIR, "| артефакт:", ARTIFACT)


## 1. Железо

In [ ]:
import torch, psutil

n = torch.cuda.device_count()
vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if n else 0.0
print(f"GPU {torch.cuda.get_device_name(0) if n else '—'} | VRAM {vram:.0f}GB | "
      f"RAM {psutil.virtual_memory().total / 1e9:.0f}GB")
assert n >= 1 and vram >= 70, (
    f"VRAM {vram:.0f} GB < 70 GB. Обучение 7B рассчитано на g2.1 (1× A100 80 GB): "
    "8-bit AdamW, gradient checkpointing и bf16 подобраны под неё. "
    "Смени конфигурацию и перезапусти VM"
)


## 2. Преполётная проверка CLI

In [ ]:
CLI = f"{REPO}/scripts/train_ft_judge.py"
assert os.path.isfile(CLI), (
    "нет scripts/train_ft_judge.py. Обучение живёт в репозитории, а не в ячейках: "
    "только так рядом с прогоном оказываются run.yaml и git-хэш. Контракт CLI — "
    "в docs/datasphere.md, раздел «FT судьи»; см. PR D3, «Требуется от других»"
)


## 3. Смоук — ~5 мин

`--limit 20`: полный путь от чтения `folds.json` до `scores.jsonl` и `degenerate_rate`,
без восьми часов ожидания.

In [ ]:
!cd {REPO} && python scripts/train_ft_judge.py --smoke-only \
    --data {DATA} --folds {FOLDS} --fold {FOLD} --repeat {REPEAT} --limit 20 \
    --model {MODEL} --mode {MODE} --tuning {TUNING} \
    --epochs 1 --max-length 512 --batch-size {BATCH_SIZE} --grad-accum 1 \
    --save-strategy {SAVE_STRATEGY} --seed {SEED} \
    --output-dir {BASE}/smoke/ft_judge \
    --predictions-output {BASE}/smoke/ft_judge/scores.jsonl


## 4. Обучение одного фолда — ~1.5 ч

LoRA на всех линейных слоях, включая MLP: `r=256`, `alpha=512`, LR ≈ 2e-4. Взвешенный
лосс и oversampling негативов обязательны — прошлый прогон на 1.5B схлопнулся в
константный вердикт (1,1) именно из-за дисбаланса 72/28, и это прошло незамеченным.

`--save-strategy epoch`: чекпоинт после каждой эпохи, обрыв сессии стоит одной эпохи,
а не всего прогона.

In [ ]:
# ~1.5 ч на A100. Прерванный прогон продолжается с последнего чекпоинта в OUTPUT_DIR.
!cd {REPO} && python scripts/train_ft_judge.py \
    --data {DATA} --folds {FOLDS} --fold {FOLD} --repeat {REPEAT} \
    --model {MODEL} --mode {MODE} \
    --tuning {TUNING} --lora-r {LORA_R} --lora-alpha {LORA_ALPHA} \
    --lora-target-modules all-linear \
    --learning-rate {LEARNING_RATE} --epochs {EPOCHS} --max-length {MAX_SEQ_LEN} \
    --batch-size {BATCH_SIZE} --grad-accum {GRAD_ACCUM} \
    --pos-weight-mode {POS_WEIGHT_MODE} --oversample-negatives \
    --save-strategy {SAVE_STRATEGY} --save-total-limit {SAVE_TOTAL_LIMIT} \
    --seed {SEED} --resume \
    --output-dir {OUTPUT_DIR} \
    --predictions-output {ARTIFACT}/scores.jsonl \
    --diagnostics-output {ARTIFACT}/ft_diagnostics.json


## 5. Контроль схлопывания — глазами

`const_share` выше 0.98 означает константный вердикт. При базовой ставке reliable 72%
такая модель показывает приличную accuracy и нулевой macro-F1 по негативному классу;
именно так прошлый прогон и выглядел «успешным».

In [ ]:
import json

diagnostics = json.load(open(f"{REPO}/{ARTIFACT}/ft_diagnostics.json", encoding="utf-8"))
print(f"{'epoch':>5} {'const_share':>12} {'entropy':>8}  degenerate")
for row in diagnostics["epochs"]:
    print(f"{row['epoch']:>5} {row['const_share']:>12.4f} "
          f"{row['output_entropy']:>8.4f}  {row['is_degenerate']}")
print()
print("collapsed:", diagnostics["collapsed"], "| причина:", diagnostics["collapse_reason"])
assert not diagnostics["collapsed"], (
    "модель схлопнулась в константный вердикт. Поднять вес негативного класса "
    "(--pos-weight-mode), усилить oversampling, снизить LR. Прогон в отчёт не идёт"
)


## 6. Оценка

Порог подбирается внутри train-части фолда, отчёт — с 95% ДИ и перцентилем шума.
Ни то, ни другое ноутбук не считает: это `scripts/evaluate_cv.py`.

In [ ]:
!cd {REPO} && python scripts/evaluate_cv.py --data {DATA} --folds {FOLDS} \
    --scores {ARTIFACT}/scores.jsonl \
    --score-expr "m3.p_faith * m3.p_rel" \
    --compare predictions/alfa/m3_judge/zero_shot/scores.jsonl \
    --output {ARTIFACT}/report.json


In [ ]:
!cd {REPO} && git add {ARTIFACT} && \
    git commit -m "results(ft_judge): фолд {FOLD}, LoRA r=256 на A100" && \
    git log --oneline -1


## 7. Экспорт чекпоинта

Чекпоинт ~15 GB лежит на File Storage и переживает рестарт VM, но не удаление
хранилища. Надёжный способ забрать — Hugging Face Hub.

In [ ]:
if PUSH_TO_HUB and HUB_MODEL_ID:
    from huggingface_hub import HfApi, create_repo

    create_repo(HUB_MODEL_ID, token=HF_TOKEN, private=True, exist_ok=True)
    HfApi().upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HUB_MODEL_ID,
        token=HF_TOKEN,
        commit_message=f"RAG judge {TUNING} Qwen2.5-7B ({MODE}, fold {FOLD})",
    )
    print("pushed:", f"https://huggingface.co/{HUB_MODEL_ID}")
else:
    print("чекпоинт:", OUTPUT_DIR)
    print("для выгрузки: PUSH_TO_HUB=True + HUB_MODEL_ID + HF_TOKEN и перезапустить ячейку")


## 8. Остальные фолды — через Jobs

Пять фолдов подряд — ~7.5 ч, VM ноутбука столько не проживёт. Конфиги заданий:

```bash
pip install datasphere
for fold in 0 1 2 3 4; do
  datasphere project job execute -p <project-id> -c jobs/ft_judge_fold${fold}.yaml
done
```

Не забыть остановить VM и деактивировать File Storage после работы — DataSphere
тарифицирует и то, и другое.